<a href="https://colab.research.google.com/github/epashkeeva-ui/project_p/blob/%D0%9A%D1%83%D0%BF%D1%87%D0%B0-%D0%A1%D0%BE%D1%84%D1%8C%D1%8F/poisk_sklada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
TWO_GIS_KEY = ""


In [ ]:
import requests
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

HEADERS = {"User-Agent": "Mozilla/5.0"}


In [ ]:
class WarehouseParser:
    #Парсит склады с сайта skladoiskatel.ru

    BASE_URL = "https://moskva.skladoiskatel.ru"
    LISTING_URL = "https://moskva.skladoiskatel.ru/moskva/holodilnye-i-morozilnye-sklady/"
    HEADERS = {"User-Agent": "Mozilla/5.0"}

    def get_links(self):
        #Ссылки на карточки складов
        response = requests.get(self.LISTING_URL, headers=self.HEADERS)
        links = re.findall(r"/sklady-ot-sobstvennikov/\d+-[a-z0-9\-]+/", response.text)
        return list(set(links))

    def parse_card(self, link):
        #Парсит одну карточку склада
        url = self.BASE_URL + link
        html = requests.get(url, headers=self.HEADERS).text
        soup = BeautifulSoup(html, "html.parser")

        obj_id = re.search(r"/(\d+)-", url).group(1)
        address = soup.find("div", class_="b-place").find("address").get_text(strip=True)

        # Цена
        price_text = soup.find("div", class_="b-meta").get_text()
        price_match = re.search(r"([\d\s,]+)\s*руб", price_text)
        if price_match:
            digits = price_match.group(1).replace(" ", "").split(",")[0]
            price = int(digits) if digits else None
        else:
            price = None

        # Координаты (на сайте 'long' = широта, 'lat' = долгота)
        lat_match = re.search(r"'long':\s*([\d.]+)", html)
        lon_match = re.search(r"'lat':\s*([\d.]+)", html)
        lat = float(lat_match.group(1)) if lat_match else None
        lon = float(lon_match.group(1)) if lon_match else None

        return {"id": obj_id, "address": address, "price": price,
                "latitude": lat, "longitude": lon}

    def parse_all(self):
        links = self.get_links()
        records = []
        for link in links:
            records.append(self.parse_card(link))
            time.sleep(0.3)
        return pd.DataFrame(records), links


parser = WarehouseParser()
df_sklado, links = parser.parse_all()
df_sklado

,id,address,price,latitude,longitude
0,121855,"Москва, Алтуфьевское шоссе ( Район: Севе...",710.0,55.864987,37.581679
1,149517,"Москва, ул. Плеханова, д.15 ( Район: Юг...",1089.0,55.749790,37.763962
2,167153,"Москва,",1000.0,55.755819,37.617644
3,118376,"Москва, 2-й Хорошевский проезд, д. 7 ( Р...",NaN,55.773622,37.530428
4,117640,"Москва, САО ( Район: Северный)",712.0,55.886267,37.500359
5,155811,"Москва, Фрязино, окружной проезд ( Район...",NaN,55.970257,38.062589
6,168001,"Москва, рябиновая 45 ( Район: Западный)",18659.0,55.697620,37.424138
7,114425,"Москва, Бусиновская ( Район: Северный)",890.0,55.880422,37.496883
8,142012,"Москва, Рябиновая, д. 45 ( Район: Западный)",12000.0,55.697620,37.424138
9,167803,"Москва, ул. Рябиновая, д. 63, стр. 1, 2 ...",1000.0,55.682645,37.433130


In [ ]:
class OzonFinder:
    #Ищет точки Ozon Fresh через 2GIS

    def __init__(self, api_key_2gis):
        self.api_key_2gis = api_key_2gis

    def find(self, query="Ozon Fresh", max_pages=10):
        results = []
        for page in range(1, max_pages + 1):
            response = requests.get(
                "https://catalog.api.2gis.com/3.0/items",
                params={
                    "q": query,
                    "region_id": 32,
                    "fields": "items.point,items.address,items.name",
                    "page_size": 10,
                    "page": page,
                    "key": self.api_key_2gis,
                }
            )
            items = response.json().get("result", {}).get("items", [])
            if not items:
                break

            for item in items:
                if "ozon" not in item.get("name", "").lower():
                    continue
                point = item.get("point", {})
                results.append({
                    "name": item["name"],
                    "address": item.get("address_name", ""),
                    "latitude": point["lat"],
                    "longitude": point["lon"],
                })
            time.sleep(0.3)

        return pd.DataFrame(results).drop_duplicates(subset=["address"])


finder = OzonFinder(api_key_2gis=TWO_GIS_KEY)
df_ozon = finder.find()
print(f"Найдено точек Ozon: {len(df_ozon)}")
df_ozon

Найдено точек Ozon: 50


,name,address,latitude,longitude
0,"Ozon fresh, склад","Молодогвардейская улица, 61 ст20",55.732148,37.394040
1,"Ozon fresh, служба доставки продуктов","Индустриальная улица, 3",55.481486,37.311420
2,"Ozon fresh, служба доставки продуктов","Полевая улица, 19",55.660230,37.252642
3,"Ozon fresh, служба доставки продуктов","Шарикоподшипниковская улица, 13 ст46",55.721584,37.684243
4,"Ozon fresh, служба доставки продуктов","проспект Мира, 211",55.846172,37.659945
5,"Ozon fresh, служба доставки продуктов","Нагорное шоссе, 4",55.894429,37.405283
6,"Ozon fresh, служба доставки продуктов","Волгоградский проспект, 1 ст1",55.733733,37.670624
7,"Ozon fresh, служба доставки продуктов","Дмитровское шоссе, 131 к1",55.888731,37.541583
8,"Ozon fresh, служба доставки продуктов","улица Свободы, 20",55.833068,37.454068
9,"Ozon fresh, служба доставки продуктов","Скандинавский бульвар, 7",55.564607,37.502581


In [ ]:
from math import radians, sin, cos, sqrt, atan2


In [ ]:
class DistanceCalculator:
    #Считает расстояния между складами и Ozon через 2GIS Routing API"""

    def __init__(self, api_key_2gis):
        self.api_key_2gis = api_key_2gis

    @staticmethod
    def haversine(lat1, lon1, lat2, lon2):
        #Расстояние по прямой между двумя точками, км"""
        R = 6371
        lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        return R * 2 * atan2(sqrt(a), sqrt(1 - a))

    def road_distance(self, lat1, lon1, lat2, lon2):
        #Расстояние по дорогам через 2GIS Routing API, км"""
        response = requests.post(
            f"https://routing.api.2gis.com/routing/7.0.0/global?key={self.api_key_2gis}",
            json={
                "points": [
                    {"type": "stop", "lat": lat1, "lon": lon1},
                    {"type": "stop", "lat": lat2, "lon": lon2},
                ],
                "transport": "driving",
            }
        )
        data = response.json()

        routes = data.get("result", [])
        if routes:
            return routes[0]["total_distance"] / 1000
        return None

    def add_distances(self, df_sklado, df_ozon):
        #Добавляет колонку distance_km — расстояние по дорогам до ближайшего Ozon"""
        df = df_sklado.copy()
        distances = []

        for i, sklado in df.iterrows():
            # Сначала Haversine ищет ближайший Ozon по прямой
            all_distances = [
                (
                    self.haversine(
                        sklado["latitude"], sklado["longitude"],
                        oz["latitude"], oz["longitude"]
                    ),
                    oz["latitude"],
                    oz["longitude"],
                )
                for _, oz in df_ozon.iterrows()
            ]
            _, nearest_lat, nearest_lon = min(all_distances)

            # 2GIS считает реальное расстояние по дорогам
            road_km = self.road_distance(
                sklado["latitude"], sklado["longitude"],
                nearest_lat, nearest_lon,
            )

            distances.append(road_km)

            if road_km is not None:
                print(f"[{i + 1}/{len(df)}] ID {sklado['id']}: {road_km:.2f} км")
            else:
                print(f"[{i + 1}/{len(df)}] ID {sklado['id']}: маршрут не построен")

            time.sleep(5)

        df["distance_km"] = distances
        return df


calc = DistanceCalculator(api_key_2gis=TWO_GIS_KEY)
df_sklado_simple = calc.add_distances(
    df_sklado[["id", "price", "latitude", "longitude"]],
    df_ozon,
)
df_sklado_simple

[1/30] ID 121855: 2.83 км
[2/30] ID 149517: 15.12 км
[3/30] ID 167153: 8.94 км
[4/30] ID 118376: 6.31 км
[5/30] ID 117640: 2.59 км
[6/30] ID 155811: 9.79 км
[7/30] ID 168001: 7.81 км
[8/30] ID 114425: 2.65 км
[9/30] ID 142012: 7.81 км
[10/30] ID 167803: 4.83 км
[11/30] ID 114380: 8.07 км
[12/30] ID 146924: 8.28 км
[13/30] ID 142013: 7.81 км
[14/30] ID 117826: 4.44 км
[15/30] ID 128690: 4.84 км
[16/30] ID 170003: 4.35 км
[17/30] ID 159233: маршрут не построен
[18/30] ID 115219: 17.81 км
[19/30] ID 107540: 2.02 км
[20/30] ID 170696: маршрут не построен
[21/30] ID 165936: 2.52 км
[22/30] ID 107448: 4.32 км
[23/30] ID 144407: 5.28 км
[24/30] ID 141157: 49.89 км
[25/30] ID 154948: 0.65 км
[26/30] ID 137000: маршрут не построен
[27/30] ID 114733: 0.82 км
[28/30] ID 107550: 8.31 км
[29/30] ID 114745: 6.92 км
[30/30] ID 114430: 7.86 км


,id,price,latitude,longitude,distance_km
0,121855,710.0,55.864987,37.581679,2.829
1,149517,1089.0,55.749790,37.763962,15.117
2,167153,1000.0,55.755819,37.617644,8.935
3,118376,NaN,55.773622,37.530428,6.314
4,117640,712.0,55.886267,37.500359,2.592
5,155811,NaN,55.970257,38.062589,9.789
6,168001,18659.0,55.697620,37.424138,7.814
7,114425,890.0,55.880422,37.496883,2.653
8,142012,12000.0,55.697620,37.424138,7.814
9,167803,1000.0,55.682645,37.433130,4.827


In [ ]:
# Дозапрашиваем маршруты только для складов с пропущенным distance_km

for i, sklado in df_sklado_simple.iterrows():
    if pd.notna(sklado["distance_km"]):
        continue

    # Haversine
    all_distances = [
        (
            calc.haversine(
                sklado["latitude"], sklado["longitude"],
                oz["latitude"], oz["longitude"]
            ),
            oz["latitude"],
            oz["longitude"],
        )
        for _, oz in df_ozon.iterrows()
    ]
    _, nearest_lat, nearest_lon = min(all_distances)

    # 2GIS по дорогам
    road_km = calc.road_distance(
        sklado["latitude"], sklado["longitude"],
        nearest_lat, nearest_lon,
    )

    df_sklado_simple.at[i, "distance_km"] = road_km

    if road_km is not None:
        print(f"ID {sklado['id']}: {road_km:.2f} км")
    else:
        print(f"ID {sklado['id']}: снова не построен")

    time.sleep(5)

df_sklado_simple

ID 159233: 6.92 км
ID 170696: 9.30 км
ID 137000: 3.66 км


,id,price,latitude,longitude,distance_km
0,121855,710.0,55.864987,37.581679,2.829
1,149517,1089.0,55.749790,37.763962,15.117
2,167153,1000.0,55.755819,37.617644,8.935
3,118376,NaN,55.773622,37.530428,6.314
4,117640,712.0,55.886267,37.500359,2.592
5,155811,NaN,55.970257,38.062589,9.789
6,168001,18659.0,55.697620,37.424138,7.814
7,114425,890.0,55.880422,37.496883,2.653
8,142012,12000.0,55.697620,37.424138,7.814
9,167803,1000.0,55.682645,37.433130,4.827


In [ ]:
df_sklado.to_csv('skladi_morozilki.csv')


In [ ]:
df_ozon.to_csv('skladi_ozon.csv')

In [ ]:
df_sklado_simple.to_csv('skladi_with_price_and_distance.csv')